Chapter 05 제출 답안 양식. 분석을 믿을 수 있게 만드는 데이터 전처리

0. 제출 정보

- 이름: 예민호

- GitHub ID: yegamanse-cpu

- 작성일: 2026.10.14

- 최종 제출 URL: 

In [29]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == 'notebooks':
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('현재 실행 위치:', CURRENT_DIR)
print('프로젝트 루트:', PROJECT_ROOT)
print('원본 데이터 폴더:', RAW_DIR)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)

현재 실행 위치: c:\dev\llm-data-analysis-course\notebooks\ch05
프로젝트 루트: c:\dev\llm-data-analysis-course\notebooks\ch05
원본 데이터 폴더: c:\dev\llm-data-analysis-course\notebooks\ch05\data\raw
전처리 데이터 폴더: c:\dev\llm-data-analysis-course\notebooks\ch05\data\processed
보고서 폴더: c:\dev\llm-data-analysis-course\notebooks\ch05\reports


In [30]:
customers = pd.read_csv(RAW_DIR / 'customers.csv')
products = pd.read_csv(RAW_DIR / 'products.csv')
orders = pd.read_csv(RAW_DIR / 'orders.csv')
order_items = pd.read_csv(RAW_DIR / 'order_items.csv')

print('원본 데이터 불러오기 완료')

원본 데이터 불러오기 완료


In [31]:
raw_data = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

raw_shapes = pd.DataFrame({
    'dataset': list(raw_data.keys()),
    'rows': [df.shape[0] for df in raw_data.values()],
    'columns': [df.shape[1] for df in raw_data.values()],
})

raw_shapes

,dataset,rows,columns
0,customers,152,6
1,products,102,4
2,orders,302,5
3,order_items,765,5


In [32]:
for name, df in raw_data.items():
    print(f'[{name}]')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print()

[customers]
shape: (152, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
shape: (102, 4)
columns: ['product_id', 'product_name', 'category', 'price']

[orders]
shape: (302, 5)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']

[order_items]
shape: (765, 5)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']



In [33]:
customers_clean = customers.copy()
products_clean = products.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()

clean_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

print('전처리용 복사본 생성 완료')

전처리용 복사본 생성 완료


In [34]:
for name, df in clean_data.items():
    print(f'\n[{name}] 결측치 개수')
    print(df.isna().sum())


[customers] 결측치 개수
customer_id    0
name           0
gender         0
age            4
city           2
signup_date    0
dtype: int64

[products] 결측치 개수
product_id      0
product_name    0
category        0
price           0
dtype: int64

[orders] 결측치 개수
order_id          0
customer_id       0
order_date        0
payment_method    0
order_status      0
dtype: int64

[order_items] 결측치 개수
order_item_id    0
order_id         0
product_id       0
quantity         0
unit_price       0
dtype: int64


In [35]:
def missing_summary(df):
    summary = pd.DataFrame({
        'missing_count': df.isna().sum(),
        'missing_ratio': (df.isna().mean() * 100).round(2),
    })
    return summary.sort_values('missing_count', ascending=False)

missing_summary(customers_clean)

,missing_count,missing_ratio
age,4,2.63
city,2,1.32
name,0,0.00
customer_id,0,0.00
gender,0,0.00
signup_date,0,0.00


1. 전처리 전 상태 기록

- 각 데이터 shape : 
    "customers" shape: (row 152, col 6), columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
    "products" shape: (row 102, col 4),columns: ['product_id', 'product_name', 'category', 'price']
    "orders" shape: (row 302, col 5), columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
    "order_items" shape: (row 765, col 5), columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']

- 결측치 : "customers" age 4, city 2 

- 전체 중복 : customers 전체 행 중복 수: 2 / products 전체 행 중복 수: 2 / orders 전체 행 중복 수: 2 / order_items 전체 행 중복 수: 2

- 주요 ID 중복 : customers customer_id 중복 수: 2 / products product_id 중복 수: 2 / orders order_id 중복 수: 2 / order_items order_item_id 중복 수: 2

- 타입 문제 후보 : quantity, unit_price (콤마나 문자 등), age (비숫자)

- 전처리 전 상태
![01 캡처](images/01.png)

- 결과 관찰 : 첫 과제시도 시 공통으로 활용되는 RAW데이터를 사용하는 실수를 했다. 그 결과 결측이나 중복이 하나도 확인되지 않아 이상함을 느끼고 실습과제 안내를 재확인하여 실습용 RAW데이터로 분석을 시행했다. 결측이 총 6건 전체 행 중복이 모두 발생했고 주요 ID도 중복이 발생해 있음을 확인하였다.  

- 나의 해석과 판단 : 금회 전처리 목적의 과제가 결측, 중복이 하나도 없을리가 없을것으로 판단했고 그 예측이 맞았었다. 결국 목적에 맞는 RAW데이터가 중요함을 반증하는 셈이다.

- 어떤 문제를 먼저 처리해야 하는지와 이유를 작성하세요 : 우선 원본데이터에 전처리를 하면 기준점이 사라져 나중에 문제가 생길수 있음으로 복사데이터를 만들어 전처리를 수행해야 한다. 

- 업무·분석적 의미 : 어떤 것을 분석하려 할 때 그에 필요한 RAW데이터가 충분히 확보되어 있는지가 분석에 첫 단계라고 판단. 

- 한계와 추가 확인 사항 : 아직 관찰단계로 타입 문제는 후보로만 적고 실제 변환 실패 건수는 변환을 돌려봐야 확정 

In [36]:
if 'age' in customers_clean.columns:
    customers_clean['age'] = pd.to_numeric(customers_clean['age'], errors='coerce')
    age_median = customers_clean['age'].median()
    customers_clean['age'] = customers_clean['age'].fillna(age_median)
    print('age 중앙값:', age_median)

if 'city' in customers_clean.columns:
    customers_clean['city'] = customers_clean['city'].fillna('Unknown')

missing_summary(customers_clean)

age 중앙값: 41.0


,missing_count,missing_ratio
customer_id,0,0.0
name,0,0.0
gender,0,0.0
age,0,0.0
city,0,0.0
signup_date,0,0.0


In [37]:
for name, df in clean_data.items():
    print(name, '전체 행 중복 수:', df.duplicated().sum())

customers 전체 행 중복 수: 2
products 전체 행 중복 수: 2
orders 전체 행 중복 수: 2
order_items 전체 행 중복 수: 2


In [38]:
key_checks = {
    'customers': ('customer_id', customers_clean),
    'products': ('product_id', products_clean),
    'orders': ('order_id', orders_clean),
    'order_items': ('order_item_id', order_items_clean),
}

for name, (key_col, df) in key_checks.items():
    if key_col in df.columns:
        print(name, key_col, '중복 수:', df[key_col].duplicated().sum())
    else:
        print(name, key_col, '컬럼 없음')

customers customer_id 중복 수: 2
products product_id 중복 수: 2
orders order_id 중복 수: 2
order_items order_item_id 중복 수: 2


In [39]:
customers_clean = customers_clean.drop_duplicates()
products_clean = products_clean.drop_duplicates()
orders_clean = orders_clean.drop_duplicates()
order_items_clean = order_items_clean.drop_duplicates()

print('완전 중복 행 제거 완료')

완전 중복 행 제거 완료


In [42]:
def strip_string_columns(df):
    result = df.copy()
    string_columns = result.select_dtypes(include='object').columns

    for col in string_columns:
        result[col] = result[col].where(
            result[col].isna(),
            result[col].astype(str).str.strip()
        )

    return result

customers_clean = strip_string_columns(customers_clean)
products_clean = strip_string_columns(products_clean)
orders_clean = strip_string_columns(orders_clean)
order_items_clean = strip_string_columns(order_items_clean)

print('문자열 앞뒤 공백 제거 완료')

문자열 앞뒤 공백 제거 완료


C:\Users\kwater\AppData\Local\Temp\ipykernel_18336\3780498526.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = result.select_dtypes(include='object').columns
C:\Users\kwater\AppData\Local\Temp\ipykernel_18336\3780498526.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/

- 실행은 되는 것으로 보이나 메세지가 발생한 이유를 llm을 활용하여 분석해보았다.

- llm의 판단으로 "Pandas는 현재 호환성을 위해 include='object' 선택에 문자열 전용 dtype인 str도 포함하고 있지만, 이 동작이 향후 바뀔 수 있다"라고 한다.

- 실행에 문제가 없고 큰 오류가 아니라 판단되므로 별도의 수정을 시행하지 않았다.

In [43]:
if 'city' in customers_clean.columns:
    display(customers_clean['city'].value_counts().head(10))

if 'order_status' in orders_clean.columns:
    display(orders_clean['order_status'].value_counts())

city
부산    23
대구    21
광주    21
서울    18
울산    17
인천    16
수원    10
성남     9
고양     8
대전     5
Name: count, dtype: int64

order_status
completed    175
cancelled     58
refunded      52
Complete       2
cancel         2
Refund         2
complete       1
COMPLETED      1
완료             1
Cancel         1
CANCELLED      1
취소             1
refund         1
REFUNDED       1
환불             1
Name: count, dtype: int64

In [44]:
status_map = {
    'complete': 'completed',
    'Complete': 'completed',
    'COMPLETED': 'completed',
    '완료': 'completed',
    'cancel': 'cancelled',
    'Cancel': 'cancelled',
    'CANCELLED': 'cancelled',
    '취소': 'cancelled',
    'refund': 'refunded',
    'Refund': 'refunded',
    'REFUNDED': 'refunded',
    '환불': 'refunded',
}

if 'order_status' in orders_clean.columns:
    orders_clean['order_status'] = orders_clean['order_status'].replace(status_map)
    display(orders_clean['order_status'].value_counts())

order_status
completed    180
cancelled     63
refunded      57
Name: count, dtype: int64

In [45]:
orders_clean['order_date'] = pd.to_datetime(orders_clean['order_date'], errors='coerce')
print('order_date 변환 실패:', orders_clean['order_date'].isna().sum())
print('주문 시작일:', orders_clean['order_date'].min())
print('주문 종료일:', orders_clean['order_date'].max())

orders_clean['order_month'] = orders_clean['order_date'].dt.to_period('M').astype(str)
orders_clean['order_dayofweek'] = orders_clean['order_date'].dt.day_name()

orders_clean[['order_date', 'order_month', 'order_dayofweek']].head()

order_date 변환 실패: 3
주문 시작일: 2025-07-01 00:00:00
주문 종료일: 2028-12-31 00:00:00


,order_date,order_month,order_dayofweek
0,2026-07-12,2026-07,Sunday
1,2025-10-02,2025-10,Thursday
2,2026-03-13,2026-03,Friday
3,2025-10-12,2025-10,Sunday
4,2026-04-17,2026-04,Friday


In [46]:
if 'signup_date' in customers_clean.columns:
    customers_clean['signup_date'] = pd.to_datetime(customers_clean['signup_date'], errors='coerce')
    print('signup_date 변환 실패:', customers_clean['signup_date'].isna().sum())
    display(customers_clean[['customer_id', 'signup_date']].head())

signup_date 변환 실패: 2


,customer_id,signup_date
0,1,2025-10-05
1,2,2023-09-20
2,3,2025-01-22
3,4,2025-03-13
4,5,2024-02-10


In [47]:
def to_number(series):
    return pd.to_numeric(
        series.astype(str).str.replace(',', '', regex=False),
        errors='coerce'
    )

products_clean['price'] = to_number(products_clean['price'])
order_items_clean['quantity'] = to_number(order_items_clean['quantity'])
order_items_clean['unit_price'] = to_number(order_items_clean['unit_price'])

print('price 변환 실패:', products_clean['price'].isna().sum())
print('quantity 변환 실패:', order_items_clean['quantity'].isna().sum())
print('unit_price 변환 실패:', order_items_clean['unit_price'].isna().sum())

price 변환 실패: 2
quantity 변환 실패: 5
unit_price 변환 실패: 2


In [48]:
display(products_clean['price'].describe())
display(order_items_clean[['quantity', 'unit_price']].describe())

count    9.800000e+01
mean     2.005102e+05
std      1.001829e+06
min     -2.000000e+04
25%      5.200000e+04
50%      9.250000e+04
75%      1.527500e+05
max      9.999999e+06
Name: price, dtype: float64

,quantity,unit_price
count,758.000000,761.000000
mean,3.168865,103494.086728
std,4.159055,56969.823769
min,-1.000000,0.000000
25%,2.000000,55000.000000
50%,3.000000,96000.000000
75%,4.000000,154000.000000
max,100.000000,197000.000000


In [49]:
print('price <= 0:', len(products_clean[products_clean['price'] <= 0]))
print('quantity <= 0:', len(order_items_clean[order_items_clean['quantity'] <= 0]))
print('unit_price <= 0:', len(order_items_clean[order_items_clean['unit_price'] <= 0]))

price <= 0: 2
quantity <= 0: 3
unit_price <= 0: 1


- price가 0 이하인 상품은 분석 대상에서 제외
- quantity가 0 이하인 주문 상세는 분석 대상에서 제외
- unit_price가 0 이하인 주문 상세는 분석 대상에서 제외

In [50]:
products_clean = products_clean[products_clean['price'] > 0]
order_items_clean = order_items_clean[order_items_clean['quantity'] > 0]
order_items_clean = order_items_clean[order_items_clean['unit_price'] > 0]

print('이상값 처리 후 products:', products_clean.shape)
print('이상값 처리 후 order_items:', order_items_clean.shape)

이상값 처리 후 products: (96, 4)
이상값 처리 후 order_items: (752, 5)


2. 문자열·타입·결측 처리

- 적용한 문자열 정리
    문자열 컬럼 앞뒤 공백 제거
    complete, Complete, COMPLETED, 완료 -> completed
    cancel, Cancel, CANCELLED, 취소 -> cancelled
    refund, Refund, REFUNDED, 환불 -> refunded

- 숫자 변환 대상과 실패 건수: price 변환 실패: 2 / quantity 변환 실패: 5 / unit_price 변환 실패: 2

- 날짜 변환 대상과 실패 건수: order_date 변환 실패: 3 / signup_date 변환 실패: 2

- 결측 처리 기준:
    고객 age 결측 4건은 중앙값(41)으로 대체
    고객 city 결측 2건은 'Unknown'으로 대체
    price가 0 이하인 상품은 분석 대상에서 제외
    quantity가 0 이하인 주문 상세는 분석 대상에서 제외
    unit_price가 0 이하인 주문 상세는 분석 대상에서 제외

![02 캡처](images/02.png)


- 결과 관찰 : 문자열 정리로 order_status가 15종 표기에서 3종으로 통일. 숫자 변환 NaN 값 9건(price 2, quantity 5, unit_price 2), 날짜변환 실패 5건(order_date 3, signup_date 2). age 결측 4건, city 결측 2건은 각각 중앙값과 Unknown으로 대체

- 나의 해석과 판단 : age는 행을 삭제하면 다른 컬럼 정보까지 함께 사라지므로 중앙값으로 대체. city는 중앙값이 없어 'Unknown' 처리. 불가피한 정보손실이 발생하나 매우 중요한 정보는 아니라고 판단된다.

- 업무·분석적 의미 : 문자열 통일을 집계 전에 하지 않으면 같은 '완료' 주문이 10여 개로 쪼개져 매출·취소율 같은 핵심 지표가 전부 틀어질 수 있다. 집계 전 처리가 매우 중요하다. 

- 한계와 추가 확인 사항 : 변환에 실패한 값들의 원본 원값을 직접 확인하지는 못함

3. 중복·범주 표준화·이상값 후보

- 제거/보류한 중복 : 완전 중복 행 2건(customers 2, products 2, orders 2, order_items 2) 및 주요 ID 중복 각 2건 완전 중복 제거

- 표준화한 범주값 : order_status 15종 표기 -> completed / cancelled / refunded 3종

- 허용값 밖 값 : price <= 0: 2 / quantity <= 0: 3 / unit_price <= 0: 1

- 이상값 후보 : price 최댓값 9,999,999(평균 약 20만, 표준편차 약 100만) / quantity 최댓값 100(평균 약 3)

![03 캡처](images/03.png)

- 결과 관찰 : 완전 중복 제거 및 0 이하 값 제외. 반면 price 9,999,999나 quantity 100은 '틀렸다'고 단정할 근거가 없어 후보로 처리

- 나의 해석과 판단 : 큰 값이 곧 오류가 아닐수도 있음. 대량 주문이 실제로 있을 수 있어 자동 삭제하면 진짜 데이터를 버리게 되므로 이상값은 삭제가 아니라 '후보'로 분리해 재확인 필요

- 업무·분석적 의미 : 이상값을 무턱대고 지우면 매출 상위 건이 사라져 분석 결과가 실제보다 축소된다. 반대로 방치하면 평균이 왜곡된다. 어느 쪽이든 판단 근거를 남겨야 한다.

- 한계와 추가 확인 사항 : 이상값 후보의 실제 주문정보(상품, 고객, 날짜)을 교차 확인하지 못함

In [51]:
order_items_clean['line_total'] = (
    order_items_clean['quantity'] * order_items_clean['unit_price']
)

order_items_clean[['quantity', 'unit_price', 'line_total']].head()

,quantity,unit_price,line_total
0,2.0,172000.0,344000.0
1,4.0,79000.0,316000.0
2,3.0,87000.0,261000.0
3,1.0,187000.0,187000.0
4,5.0,125000.0,625000.0


In [52]:
print('전처리 후 주문 상세 금액 합계:', order_items_clean['line_total'].sum())

전처리 후 주문 상세 금액 합계: 252140000.0


In [53]:
invalid_customers = orders_clean[
    ~orders_clean['customer_id'].isin(customers_clean['customer_id'])
]

invalid_orders = order_items_clean[
    ~order_items_clean['order_id'].isin(orders_clean['order_id'])
]

invalid_products = order_items_clean[
    ~order_items_clean['product_id'].isin(products_clean['product_id'])
]

relationship_checks = pd.DataFrame({
    'check': [
        'orders.customer_id exists in customers.customer_id',
        'order_items.order_id exists in orders.order_id',
        'order_items.product_id exists in products.product_id',
    ],
    'invalid_count': [
        len(invalid_customers),
        len(invalid_orders),
        len(invalid_products),
    ],
})

relationship_checks

,check,invalid_count
0,orders.customer_id exists in customers.custome...,2
1,order_items.order_id exists in orders.order_id,2
2,order_items.product_id exists in products.prod...,26


In [54]:
processed_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

processed_shapes = pd.DataFrame({
    'dataset': list(processed_data.keys()),
    'rows': [df.shape[0] for df in processed_data.values()],
    'columns': [df.shape[1] for df in processed_data.values()],
})

comparison = raw_shapes.merge(
    processed_shapes,
    on='dataset',
    suffixes=('_raw', '_processed')
)

comparison

,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,152,6,150,6
1,products,102,4,96,4
2,orders,302,5,300,7
3,order_items,765,5,752,6


4. PK/FK와 파생 컬럼 재검증

- 없는 customer_id : 2건 (orders customer_id -> customers 없음)

- 없는 order_id : 2건 (order_items order_id -> orders 없음)

- 없는 product_id : 26건 (order_items product_id -> products 없음)

- 생성한 파생 컬럼 : order_month, order_dayofweek (orders) / line_total (order_items)

- line_total 검증 : quantity × unit_price, 합계 252,140,000

![04 캡처](images/04.png)

- 결과 관찰 : 전처리 후에도 키가 없는 건이 많음. 이중 product_id 미존재가 26건으로 price <= 0 상품을 제외하여 파생된 것으로 판단됨

- 나의 해석과 판단 : 테이블에서 행을 지우면 다른 참조 테이블이 깨질 수 있어 전처리 후 PK/FK 검증 과정 필요

- 업무·분석적 의미 : FK 깨진 후 테이블을 조인하면 집계가 어긋나므로 검증이 반드시 필요함

5. 전처리 전/후 비교

| 항목 | 처리 전 | 처리 후 | 변화 이유 |
| --- | --- | --- | --- |
| 행 수 | customers 152 / products 102 / orders 302 / order_items 765 | customers 150 / products 96 / orders 300 / order_items 752 | 완전 중복 제거, 0 이하·변환 실패 값 제외 |
| 결측 | customers age 4, city 2 | 0 | age 중앙값 대체, city 'Unknown' 처리 |
| 중복 | 각 데이터 2건 | 0 | 완전 중복 행 제거 |
| 변환 실패 | price 2 / quantity 5 / unit_price 2 / order_date 3 / signup_date 2 | NaN 처리 | 숫자·날짜 변환으로 NaN 처리  |

![05 캡처](images/05.png)

- 결과 관찰 : 행 수 감소 폭이 customers·orders는 중복 2건만 줄었고, products 6건과 order_items 13건은 중복에 더해 0 이하 값과 변환 실패 값까지 빠지면서 더 많이 줄음

- 나의 해석과 판단 : 행 수 변화는 중복 제거, 0 이하 제외, 변환 실패 처리로 설명되어 숫자 이상이나 설명 안 되는 감소 없었음

- 업무·분석적 의미 : 전/후 비교표는 분석 결과에 대한 신뢰의 근거가 될 수 있음

- 한계와 추가 확인 사항 : 제외한 행이 분석 결론에 영향을 줄 만큼 중요한 건이었는지는 별도로 확인하지 못함

In [55]:
customers_clean.to_csv(PROCESSED_DIR / 'customers_clean.csv', index=False, encoding='utf-8-sig')
products_clean.to_csv(PROCESSED_DIR / 'products_clean.csv', index=False, encoding='utf-8-sig')
orders_clean.to_csv(PROCESSED_DIR / 'orders_clean.csv', index=False, encoding='utf-8-sig')
order_items_clean.to_csv(PROCESSED_DIR / 'order_items_clean.csv', index=False, encoding='utf-8-sig')

list(PROCESSED_DIR.glob('*_clean.csv'))

[WindowsPath('c:/dev/llm-data-analysis-course/notebooks/ch05/data/processed/customers_clean.csv'),
 WindowsPath('c:/dev/llm-data-analysis-course/notebooks/ch05/data/processed/orders_clean.csv'),
 WindowsPath('c:/dev/llm-data-analysis-course/notebooks/ch05/data/processed/order_items_clean.csv'),
 WindowsPath('c:/dev/llm-data-analysis-course/notebooks/ch05/data/processed/products_clean.csv')]

In [56]:
summary_text = f'''# Chapter 5 데이터 전처리 요약

## 전처리 결과 파일

- customers_clean.csv
- products_clean.csv
- orders_clean.csv
- order_items_clean.csv

## 전처리 전후 데이터 크기

```text
{comparison.to_string(index=False)}
```

## 파일 간 관계 점검 결과

```text
{relationship_checks.to_string(index=False)}
```

## 주요 처리 내용

- 원본 데이터를 직접 수정하지 않고 복사본을 사용함
- 문자열 컬럼 앞뒤 공백 제거
- 고객 나이 결측치는 중앙값으로 대체
- 고객 도시 결측치는 Unknown으로 처리
- 주문 상태값 표기 통일
- 날짜 컬럼 변환 및 주문 월/요일 파생 컬럼 생성
- 가격, 수량, 단가를 숫자형으로 변환
- 0 이하 가격, 수량, 단가 확인 및 처리
- line_total 파생 컬럼 생성
- 파일 간 키 관계 재확인

## 주의 사항

이 전처리 기준은 실습용 예시입니다. 실제 업무에서는 결측치와 이상값을 삭제하거나 대체하기 전에 원본 시스템, 수집 과정, 업무 담당자 확인이 필요합니다.
'''

report_path = REPORT_DIR / 'ch05_preprocessing_summary.md'
report_path.write_text(summary_text, encoding='utf-8')

print('요약 보고서 저장 완료:', report_path)

요약 보고서 저장 완료: c:\dev\llm-data-analysis-course\notebooks\ch05\reports\ch05_preprocessing_summary.md


In [57]:
def preprocess_customers(df):
    result = strip_string_columns(df)

    if 'age' in result.columns:
        result['age'] = pd.to_numeric(result['age'], errors='coerce')
        result['age'] = result['age'].fillna(result['age'].median())

    if 'city' in result.columns:
        result['city'] = result['city'].fillna('Unknown')

    if 'signup_date' in result.columns:
        result['signup_date'] = pd.to_datetime(result['signup_date'], errors='coerce')

    return result.drop_duplicates()


def preprocess_products(df):
    result = strip_string_columns(df)

    if 'price' in result.columns:
        result['price'] = to_number(result['price'])
        result = result[result['price'] > 0]

    return result.drop_duplicates()


def preprocess_orders(df):
    result = strip_string_columns(df)

    if 'order_status' in result.columns:
        result['order_status'] = result['order_status'].replace(status_map)

    if 'order_date' in result.columns:
        result['order_date'] = pd.to_datetime(result['order_date'], errors='coerce')
        result['order_month'] = result['order_date'].dt.to_period('M').astype(str)
        result['order_dayofweek'] = result['order_date'].dt.day_name()

    return result.drop_duplicates()


def preprocess_order_items(df):
    result = strip_string_columns(df)

    if 'quantity' in result.columns:
        result['quantity'] = to_number(result['quantity'])
        result = result[result['quantity'] > 0]

    if 'unit_price' in result.columns:
        result['unit_price'] = to_number(result['unit_price'])
        result = result[result['unit_price'] > 0]

    if {'quantity', 'unit_price'}.issubset(result.columns):
        result['line_total'] = result['quantity'] * result['unit_price']

    return result.drop_duplicates()

In [58]:
customers_clean_func = preprocess_customers(customers)
products_clean_func = preprocess_products(products)
orders_clean_func = preprocess_orders(orders)
order_items_clean_func = preprocess_order_items(order_items)

print(customers_clean_func.shape)
print(products_clean_func.shape)
print(orders_clean_func.shape)
print(order_items_clean_func.shape)

(150, 6)
(96, 4)
(300, 7)
(752, 6)


C:\Users\kwater\AppData\Local\Temp\ipykernel_18336\3780498526.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = result.select_dtypes(include='object').columns
C:\Users\kwater\AppData\Local\Temp\ipykernel_18336\3780498526.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/

In [59]:
from src.data_loader import load_sales_data
from src.preprocessing import (
    build_preprocessing_report,
    compare_shapes,
    duplicate_summary,
    preprocess_sales_data,
    save_processed_data,
    validate_relationships,
)

module_raw_data = load_sales_data(RAW_DIR)
module_processed_data = preprocess_sales_data(module_raw_data)
module_relationship_checks = validate_relationships(module_processed_data)
module_comparison = compare_shapes(module_raw_data, module_processed_data)

module_comparison

,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,152,6,150,6
1,order_items,765,5,752,6
2,orders,302,5,300,7
3,products,102,4,96,4


In [60]:
duplicate_summary(
    module_processed_data,
    key_columns={
        'customers': 'customer_id',
        'products': 'product_id',
        'orders': 'order_id',
        'order_items': 'order_item_id',
    },
)

,dataset,row_duplicate_count,key_column,key_duplicate_count
0,customers,0,customer_id,0
1,products,0,product_id,0
2,orders,0,order_id,0
3,order_items,0,order_item_id,0


In [61]:
module_relationship_checks

,check,invalid_count
0,orders.customer_id exists in customers.custome...,2
1,order_items.order_id exists in orders.order_id,2
2,order_items.product_id exists in products.prod...,26


6. 재현 가능한 전처리 확인

- 생성된 clean CSV : customers_clean.csv / products_clean.csv / orders_clean.csv / order_items_clean.csv

- 생성된 요약 보고서 : reports/ch05_preprocessing_summary.md

- 재실행 결과 : src.preprocessing 결과 동일(customers 150, products 96, orders 300, order_items 752 / 중복·키 중복 모두 0)

- 재실행 후 달라진 점 : 없음. 

![06 캡처](images/06.png)

- 나의 해석과 판단 : 전처리 로직을 함수·스크립트로 묶으면 같은 결과가 나오므로 언제나 재현가능한 값을 만들 수 있음

7. Chapter 05 최종 판단

- 내가 정한 전처리 원칙 3가지

    1. 원본은 건드리지 않고 복사본에서만 처리
    2. 삭제보다 보존을 고려. 결측·이상값은 바로 지우지 말고 대체하거나 후보로 분리
    3. 모든 처리에는 근거를 남기며 무엇을 왜 바꿨는지 설명할 수 없으면 처리하지 않음

- 가장 위험하다고 생각한 자동 처리 1가지와 이유 : 이상값의 자동 삭제. 값이 크다는 이유로 삭제하면 실제 있을 수 있는 진짜 데이터를 잃고, 데이터가 과소 집계될 수 있음

- 다음 EDA에서 특히 주의할 데이터 특성 : price·quantity에 남아 있는 큰 값 후보, 전처리 후에도 남은 깨진 FK(product_id 26건 등)로 조인 누락

- 현재 전처리의 한계 : 변환 실패·이상값 발생 이유를 살펴보지 못하고 가이드를 따라 분석 시행

최종 제출 체크
- [X] raw 원본을 덮어쓰지 않았습니다.
- [X] 처리 전/후 비교를 남겼습니다.
- [X] 처리 기준과 이유를 자신의 말로 작성했습니다.
- [X] clean CSV와 재실행 결과를 확인했습니다.
- [X] 개인정보/Secret이 없습니다.
- [X] `chapter05/chapter05.ipynb`가 GitHub에서 정상 표시됩니다.
- [X] 최종 Notebook 파일 URL을 제출합니다.